In [1]:
##for generating the embeddings

In [2]:
import gensim.downloader as api
import numpy as np

model = api.load("glove-wiki-gigaword-50")  # only ~70MB

def sentence_embedding(text):
    words = text.split()
    vectors = [model[w] for w in words if w in model]
    return np.mean(vectors, axis=0)

vec = sentence_embedding("I love AI")
print(len(vec))  # 50-dim

[==================================================] 100.0% 66.0/66.0MB downloaded
50


In [3]:
vec

array([-0.13886  ,  1.1401   , -0.85212  , -0.29212  ,  0.75534  ,
        0.82762  , -0.3181   ,  0.0072204, -0.34762  ,  1.0731   ,
       -0.24665  ,  0.97765  , -0.55835  , -0.090318 ,  0.83182  ,
       -0.33317  ,  0.22648  ,  0.30913  ,  0.026929 , -0.086739 ,
       -0.14703  ,  1.3543   ,  0.53695  ,  0.43735  ,  1.2749   ,
       -1.4382   , -1.2815   , -0.15196  ,  1.0506   , -0.93644  ,
        2.7561   ,  0.58967  , -0.29473  ,  0.27574  , -0.32928  ,
       -0.201    , -0.28547  , -0.45987  , -0.14603  , -0.69372  ,
        0.070761 , -0.19326  , -0.1855   , -0.16095  ,  0.24268  ,
        0.20784  ,  0.030924 , -1.3711   , -0.28606  ,  0.2898   ],
      dtype=float32)

In [9]:
laks=sentence_embedding("i love english and i is very good subject")
laks

array([ 3.7760004e-02,  4.2185497e-01, -6.2355733e-01, -3.6931145e-01,
        5.3862262e-01,  1.6314800e-01, -3.1964055e-01, -2.3278412e-03,
       -4.6985853e-01,  1.9722691e-01,  3.2274447e-02,  6.0801440e-01,
       -3.3763325e-01, -2.6349032e-01,  5.7380748e-01,  1.6456109e-01,
        2.7229977e-01,  2.4325667e-01, -1.4111003e-01, -4.1119546e-01,
       -2.7281001e-01,  5.2046776e-01,  4.1131732e-01,  3.5037515e-01,
        7.7355325e-01, -1.7781554e+00, -7.9907113e-01,  1.5805298e-01,
        4.6611172e-01, -3.4435266e-01,  3.4865777e+00,  9.4886668e-02,
       -1.3460778e-01, -2.3946172e-01,  6.1828449e-02, -7.9128534e-02,
        5.8615778e-02,  4.0010354e-01, -4.2748895e-02, -3.3618888e-01,
        1.0728094e-01,  6.4551912e-02, -3.9238885e-02,  2.8454652e-01,
       -5.4050088e-03,  1.4134200e-01, -6.0594227e-02, -2.4239723e-01,
       -1.8560002e-02,  4.2077887e-01], dtype=float32)

In [12]:
##we have to load the data
from langchain_community.document_loaders import TextLoader
loader=TextLoader("E:\GenerativeAI\Langchain\DataIngestion\data.txt")

<>:3: SyntaxWarning: invalid escape sequence '\G'
<>:3: SyntaxWarning: invalid escape sequence '\G'
C:\Users\PC\AppData\Local\Temp\ipykernel_15760\1758007405.py:3: SyntaxWarning: invalid escape sequence '\G'
  loader=TextLoader("E:\GenerativeAI\Langchain\DataIngestion\data.txt")


In [13]:
data=loader.load()

In [14]:
data

[Document(metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='I love learning machine learning. , general\nArtificial intelligence is the future. , general\nChatGPT helps in coding and problem solving. , general\nPython is a powerful programming language. , general\nData science involves statistics and programming. , general\nDeep learning is a subset of machine learning. , general\nNatural language processing deals with text data. , general\n\nI really enjoyed this movie, it was fantastic! , positive\nThe product quality is terrible and disappointing. , negative\nAmazing experience, I would definitely recommend it. , positive\nWorst service ever, I am not happy at all. , negative\nThe food was okay, not great but not bad either. , neutral\nAbsolutely loved it! , positive\nI hate this app, it crashes every time. , negative\n\nUser: Hi, how are you? , conversation\nBot: I am fine, how can I help you? , conversation\nUser: What is machine learning? 

In [15]:
##after loading the data we have to do the textsplitting and convert into chunks 
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=50)

In [21]:
docs=text_splitter.split_documents(data)

In [23]:
docs

[Document(metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='I love learning machine learning. , general\nArtificial intelligence is the future. , general\nChatGPT helps in coding and problem solving. , general\nPython is a powerful programming language. , general'),
 Document(metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='Data science involves statistics and programming. , general\nDeep learning is a subset of machine learning. , general\nNatural language processing deals with text data. , general'),
 Document(metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='I really enjoyed this movie, it was fantastic! , positive\nThe product quality is terrible and disappointing. , negative\nAmazing experience, I would definitely recommend it. , positive'),
 Document(metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='Worst service ever

In [25]:
from langchain_community.vectorstores import FAISS

In [31]:
db=FAISS.from_documents(docs,sentence_embedding)

AttributeError: 'function' object has no attribute 'embed_documents'

In [32]:
from langchain.embeddings.base import Embeddings
import numpy as np

class GensimEmbeddings(Embeddings):
    def __init__(self, model):
        self.model = model

    def embed_documents(self, texts):
        return [self._embed(text) for text in texts]

    def embed_query(self, text):
        return self._embed(text)

    def _embed(self, text):
        words = text.split()
        vectors = [self.model[w] for w in words if w in self.model]
        
        if len(vectors) == 0:
            return [0.0] * self.model.vector_size
        
        return np.mean(vectors, axis=0).tolist()

In [35]:


embedding = GensimEmbeddings(model)

db = FAISS.from_documents(docs, embedding)

In [36]:
db

In [37]:
top_k=db.similarity_search("What is AI ?")
top_k


[Document(id='3415de61-740f-4d17-a166-ace8e9581f42', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='Worst service ever, I am not happy at all. , negative\nThe food was okay, not great but not bad either. , neutral\nAbsolutely loved it! , positive\nI hate this app, it crashes every time. , negative'),
 Document(id='af051699-ce38-47cf-924c-2d51519446b4', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='User: Hi, how are you? , conversation\nBot: I am fine, how can I help you? , conversation\nUser: What is machine learning? , conversation'),
 Document(id='67532650-df43-46cb-84f1-2e9b0ba36628', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='User: What is machine learning? , conversation\nBot: Machine learning is a field of AI that allows systems to learn from data. , conversation\nUser: Can you help me with Python? , conversation'),
 Document(id='8ee1336d-d5ac-4

In [39]:
top_k[1].page_content

'User: Hi, how are you? , conversation\nBot: I am fine, how can I help you? , conversation\nUser: What is machine learning? , conversation'

In [46]:
##second method to searching any query is as retriever
retriever=db.as_retriever()
retriever.invoke("what is ai")

[Document(id='3415de61-740f-4d17-a166-ace8e9581f42', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='Worst service ever, I am not happy at all. , negative\nThe food was okay, not great but not bad either. , neutral\nAbsolutely loved it! , positive\nI hate this app, it crashes every time. , negative'),
 Document(id='34c586b3-b4cd-44ea-9d63-69bd5c2a2045', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='I really enjoyed this movie, it was fantastic! , positive\nThe product quality is terrible and disappointing. , negative\nAmazing experience, I would definitely recommend it. , positive'),
 Document(id='af051699-ce38-47cf-924c-2d51519446b4', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='User: Hi, how are you? , conversation\nBot: I am fine, how can I help you? , conversation\nUser: What is machine learning? , conversation'),
 Document(id='67532650-df43-46cb-84f

In [43]:
print(top_m)

[Document(id='3415de61-740f-4d17-a166-ace8e9581f42', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='Worst service ever, I am not happy at all. , negative\nThe food was okay, not great but not bad either. , neutral\nAbsolutely loved it! , positive\nI hate this app, it crashes every time. , negative'), Document(id='34c586b3-b4cd-44ea-9d63-69bd5c2a2045', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='I really enjoyed this movie, it was fantastic! , positive\nThe product quality is terrible and disappointing. , negative\nAmazing experience, I would definitely recommend it. , positive'), Document(id='af051699-ce38-47cf-924c-2d51519446b4', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='User: Hi, how are you? , conversation\nBot: I am fine, how can I help you? , conversation\nUser: What is machine learning? , conversation'), Document(id='67532650-df43-46cb-84f1-2

In [48]:
vec=embedding.embed_query("what is ai")

In [49]:
db.similarity_search_by_vector(vec)

[Document(id='3415de61-740f-4d17-a166-ace8e9581f42', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='Worst service ever, I am not happy at all. , negative\nThe food was okay, not great but not bad either. , neutral\nAbsolutely loved it! , positive\nI hate this app, it crashes every time. , negative'),
 Document(id='34c586b3-b4cd-44ea-9d63-69bd5c2a2045', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='I really enjoyed this movie, it was fantastic! , positive\nThe product quality is terrible and disappointing. , negative\nAmazing experience, I would definitely recommend it. , positive'),
 Document(id='af051699-ce38-47cf-924c-2d51519446b4', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='User: Hi, how are you? , conversation\nBot: I am fine, how can I help you? , conversation\nUser: What is machine learning? , conversation'),
 Document(id='67532650-df43-46cb-84f

In [51]:
db.save_local("faiss_index")

In [55]:
new_db=FAISS.load_local("faiss_index",embedding,allow_dangerous_deserialization=True)

In [56]:
new_db.similarity_search("what is machine learning")

[Document(id='67532650-df43-46cb-84f1-2e9b0ba36628', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='User: What is machine learning? , conversation\nBot: Machine learning is a field of AI that allows systems to learn from data. , conversation\nUser: Can you help me with Python? , conversation'),
 Document(id='af051699-ce38-47cf-924c-2d51519446b4', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='User: Hi, how are you? , conversation\nBot: I am fine, how can I help you? , conversation\nUser: What is machine learning? , conversation'),
 Document(id='468e6276-4bd5-42f4-8797-353a4e13797e', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='Machine learning is a rapidly growing field in computer science. It focuses on building systems that can learn from data and improve their performance over time without being explicitly programmed. ,'),
 Document(id='11222608-0f40-

In [ ]:
new_db.similarity_search_with_relevance_scores("what is machine learning")

C:\Users\PC\AppData\Local\Temp\ipykernel_15760\2081341795.py:1: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='67532650-df43-46cb-84f1-2e9b0ba36628', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='User: What is machine learning? , conversation\nBot: Machine learning is a field of AI that allows systems to learn from data. , conversation\nUser: Can you help me with Python? , conversation'), np.float32(-0.06854439)), (Document(id='af051699-ce38-47cf-924c-2d51519446b4', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='User: Hi, how are you? , conversation\nBot: I am fine, how can I help you? , conversation\nUser: What is machine learning? , conversation'), np.float32(-0.6087744)), (Document(id='468e6276-4bd5-42f4-8797-353a4e13797e', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='Machine learning is a rapidly growing field in computer 

[(Document(id='67532650-df43-46cb-84f1-2e9b0ba36628', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='User: What is machine learning? , conversation\nBot: Machine learning is a field of AI that allows systems to learn from data. , conversation\nUser: Can you help me with Python? , conversation'),
  np.float32(-0.06854439)),
 (Document(id='af051699-ce38-47cf-924c-2d51519446b4', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='User: Hi, how are you? , conversation\nBot: I am fine, how can I help you? , conversation\nUser: What is machine learning? , conversation'),
  np.float32(-0.6087744)),
 (Document(id='468e6276-4bd5-42f4-8797-353a4e13797e', metadata={'source': 'E:\\GenerativeAI\\Langchain\\DataIngestion\\data.txt'}, page_content='Machine learning is a rapidly growing field in computer science. It focuses on building systems that can learn from data and improve their performance over time without bein